# TikHarm fusion experiments: does audio help, and when?

Same frozen embeddings and the same **creator-grouped split** as the baseline. Only small heads are trained.

| Method | Idea |
|---|---|
| video only / audio only | single-modality heads (the comparison point) |
| early fusion | concatenate pooled video + audio features, one classifier |
| late fusion | combine the video-only and audio-only probabilities, weight tuned on val |
| **gated fusion (router)** | a learned gate scales the audio branch using `music_p`, `speech_p`, `aud_ok` |
| cross-attention | video tokens attend to audio tokens |
| (no mod-drop) variants | the same models trained without random audio dropout |

**Controls for every fusion model** (test time): real audio, audio removed, audio from a clip of a *different class*.
If scores don't change when audio is removed or swapped, the model is not using the audio.

Everything is averaged over 5 seeds and reported with 95% bootstrap intervals and paired differences against video-only.
Nothing here displays clips.

In [1]:
# Cell 1: load features, split, near-duplicate exclusion
import os, re, glob, json, time
import numpy as np, pandas as pd
import torch, torch.nn as nn
from sklearn.metrics import f1_score, roc_auc_score, recall_score

from sklearn.model_selection import StratifiedGroupKFold

def creator_of(path):
    b = os.path.splitext(os.path.basename(path))[0]; m = re.match(r"^(.*)_(\d{12,20})$", b)
    return (m.group(1) if m else b).lower()

def duration_match(d, bins, seed=0):
    d = d.copy(); d["dbin"] = pd.cut(d.dur, bins, labels=False, include_lowest=True); parts = []
    for (sp, b), g in d.groupby(["split", "dbin"]):
        c = g.label.value_counts()
        if len(c) < 4: continue
        n_ = int(c.min()); parts += [g[g.label == k].sample(n_, random_state=seed) for k in range(4)]
    return pd.concat(parts) if parts else d.iloc[:0]

def ensure_grouped(order, man, bins=(0, 10, 20, 40, 90, 1e9), seed=0):
    """If any creator appears in both train/val and test, apply the deterministic creator-grouped re-split
    (same code and seed as the baseline, so every notebook gets the identical split). Row order is unchanged."""
    if "path" not in man:
        print("manifest has no path column: cannot check creators"); return order
    cr = man.path.map(creator_of).values; sp = order.split.values
    if not (set(cr[sp == "test"]) & set(cr[np.isin(sp, ["train", "val"])])):
        print("creator check passed: the split is already creator-grouped"); return order
    print("OFFICIAL split detected (creators overlap): applying the creator-grouped re-split (seed 0)")
    y = order.label.values
    rest, test = next(iter(StratifiedGroupKFold(5, shuffle=True, random_state=seed).split(np.zeros(len(order)), y, cr)))
    tr_i, va_i = next(iter(StratifiedGroupKFold(8, shuffle=True, random_state=seed + 1).split(np.zeros(len(rest)), y[rest], cr[rest])))
    new = np.array(["train"] * len(order), dtype=object); new[rest[va_i]] = "val"; new[test] = "test"
    matched = duration_match(pd.DataFrame({"uid": order.uid.values, "dur": man.dur.values, "label": y, "split": new}), list(bins), seed)
    o2 = order.copy(); o2["split"] = new; o2["in_matched"] = o2.uid.isin(matched.uid).astype(int)
    assert not (set(cr[new == "test"]) & set(cr[new != "test"])), "re-split failed"
    print(pd.crosstab(o2.cls, o2.split)); return o2


# Embeddings are in Kaggle INPUT (read-only)
EMB = "/kaggle/input/datasets/sanawerb/emb-folder/kaggle/working/emb"

# Preparation/manifest is also in Kaggle INPUT
PREP = "/kaggle/input/datasets/sanawerb/preparation"

# Results from the previous notebook are also in Kaggle INPUT
PREV_RES = "/kaggle/input/datasets/sanawerb/results/kaggle/working/results"
WORK = os.environ.get("WORK_DIR", "/kaggle/working")
RES = f"{WORK}/results"; os.makedirs(RES, exist_ok=True)
CFG = dict(d=256, drop=0.4, mod_drop=0.3, epochs=int(os.environ.get("EPOCHS", 40)), lr=1e-3, wd=0.05, bs=64, smooth=0.1,
           seeds=list(range(int(os.environ.get("NSEEDS", 5)))), nboot=int(os.environ.get("NBOOT", 1000)))
CLASS_NAMES = ["Safe", "Harmful", "Adult", "Suicide"]
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", dev)

order = pd.read_csv(f"{EMB}/order.csv")
files = sorted(f for f in glob.glob(f"{EMB}/chunk_*.npz") if re.search(r"chunk_\d{3}\.npz$", f))
parts = [np.load(f, allow_pickle=True) for f in files]
Fz = {k: np.concatenate([p[k] for p in parts]) for k in parts[0].files}
assert list(Fz["uid"]) == list(order.uid[: len(Fz["uid"])]), "chunks do not match order.csv"
N = len(Fz["uid"]); order = order.iloc[:N].copy()
man = pd.read_csv(f"{PREP}/manifest_full (1).csv").set_index("uid").loc[order.uid].reset_index()
order = ensure_grouped(order, man)

V = Fz["vit"].astype(np.float32)               # (N, 8, 768)
A = Fz["ast"].astype(np.float32)               # (N, 5, 768)
aud_ok = Fz["aud_ok"].astype(np.float32)
music_p = man["music_p"].values.astype(np.float32) if "music_p" in man else np.zeros(N, np.float32)
speech_p = man["speech_p"].values.astype(np.float32) if "speech_p" in man else np.zeros(N, np.float32)
G = np.stack([music_p, speech_p, aud_ok], 1).astype(np.float32)           # gate inputs
y = order.label.values; split = order.split.values.copy(); in_matched = order.in_matched.values.astype(bool)

# exclude test clips with a near-identical train/val neighbour (cosine > 0.95 on ViT mean embeddings)
vm = V.mean(1); vm /= np.linalg.norm(vm, axis=1, keepdims=True) + 1e-8
te_i, tr_i = np.where(split == "test")[0], np.where(split != "test")[0]
leak = te_i[(vm[te_i] @ vm[tr_i].T).max(1) > 0.95]
split[leak] = "excluded"; print("excluded near-duplicate test clips:", len(leak))


def creator_of(path):
    b = os.path.splitext(os.path.basename(path))[0]; m = re.match(r"^(.*)_(\d{12,20})$", b)
    return (m.group(1) if m else b).lower()

def assert_creator_disjoint(paths, split_arr):
    cr = pd.Series(paths).map(creator_of).values
    shared = set(cr[split_arr == "test"]) & set(cr[np.isin(split_arr, ["train", "val"])])
    assert not shared, (f"{len(shared)} creators appear in both train/val and test. This is the OFFICIAL split, not the "
                        "creator-grouped one. Restore the grouped order.csv (re-split cell) before running.")
    print("creator check passed: no creator appears in both train/val and test")
if "path" in man: assert_creator_disjoint(man.path.values, split)

grp = np.full(N, "other audio", dtype=object)
if "music_dominant" in man: grp[man.music_dominant.values == 1] = "music-dominant"
grp[(speech_p > 0.5) & (grp != "music-dominant")] = "speech present"
print(pd.crosstab(order.cls, [split, in_matched]))
print(pd.Series(grp).value_counts().to_dict())

device: cuda
OFFICIAL split detected (creators overlap): applying the creator-grouped re-split (seed 0)
split            test  train  val
cls                              
Adult Content     218    662   68
Harmful Content   182    669  128
Safe              233    684   70
Suicide           238    669   70
excluded near-duplicate test clips: 1
creator check passed: no creator appears in both train/val and test
col_0           excluded  test       train         val      
col_1              True  False True  False True  False True 
cls                                                         
Adult Content          1   119    98   411   251    42    26
Harmful Content        0    83    99   418   251   102    26
Safe                   0   134    99   433   251    44    26
Suicide                0   139    99   418   251    44    26
{'music-dominant': 2117, 'speech present': 1292, 'other audio': 482}


In [2]:
# Cell 2: metrics, bootstrap, paired differences
def metrics(yt, P):
    pred = P.argmax(1)
    m = dict(macro_F1=f1_score(yt, pred, average="macro"),
             macro_AUC=roc_auc_score(yt, P, multi_class="ovr", average="macro", labels=[0, 1, 2, 3]),
             top2_acc=np.mean([yt[i] in np.argsort(-P[i])[:2] for i in range(len(yt))]))
    rec = recall_score(yt, pred, average=None, labels=[0, 1, 2, 3], zero_division=0)
    m.update({f"rec_{c}": r for c, r in zip(CLASS_NAMES, rec)})
    return m

def with_ci(yt, P, nboot=None):
    nboot = nboot or CFG["nboot"]; pt = metrics(yt, P); rng = np.random.default_rng(0); b = []
    for _ in range(nboot):
        ix = rng.integers(0, len(yt), len(yt))
        if len(np.unique(yt[ix])) == 4: b.append(metrics(yt[ix], P[ix]))
    b = pd.DataFrame(b)
    return pt, {k: (np.percentile(b[k], 2.5), np.percentile(b[k], 97.5)) for k in pt}

def fmt(pt, ci, k): return f"{pt[k]:.3f} [{ci[k][0]:.3f}, {ci[k][1]:.3f}]"

def paired(yt, Pbase, Pother, nboot=None):
    nboot = nboot or CFG["nboot"]; rng = np.random.default_rng(1); d = []
    for _ in range(nboot):
        ix = rng.integers(0, len(yt), len(yt))
        d.append(f1_score(yt[ix], Pother[ix].argmax(1), average="macro") - f1_score(yt[ix], Pbase[ix].argmax(1), average="macro"))
    d = np.array(d); return d.mean(), np.percentile(d, 2.5), np.percentile(d, 97.5)

In [3]:
# Cell 3: the models
class Net(nn.Module):
    """mode: video | audio | early | gate | xattn.  v:(B,8,768) a:(B,5,768) m:(B,) audio available  g:(B,3) gate inputs"""
    def __init__(s, mode, d=256, p=0.4, n_cls=4):
        super().__init__(); s.mode = mode
        mk = lambda: nn.Sequential(nn.LayerNorm(768), nn.Dropout(p * 0.5), nn.Linear(768, d), nn.GELU(), nn.Dropout(p))
        s.vp, s.ap = mk(), mk()
        s.gate = nn.Sequential(nn.Linear(3, 16), nn.GELU(), nn.Linear(16, 1))
        s.attn = nn.MultiheadAttention(d, 4, dropout=0.1, batch_first=True); s.an = nn.LayerNorm(d)
        din = {"video": d, "audio": d, "early": 2 * d + 1, "gate": 2 * d, "xattn": d}[mode]
        s.head = nn.Sequential(nn.Dropout(p), nn.Linear(din, n_cls))

    def gate_value(s, m, g):
        return torch.sigmoid(s.gate(g)).squeeze(-1) * m

    def forward(s, v, a, m, g):
        if s.mode == "audio":
            return s.head(s.ap(a).mean(1) * m[:, None])
        hv = s.vp(v)
        if s.mode == "video":
            return s.head(hv.mean(1))
        ha = s.ap(a)
        if s.mode == "early":
            return s.head(torch.cat([hv.mean(1), ha.mean(1) * m[:, None], m[:, None]], 1))
        if s.mode == "gate":
            return s.head(torch.cat([hv.mean(1), ha.mean(1) * s.gate_value(m, g)[:, None]], 1))
        att, _ = s.attn(hv, ha, ha)                              # video tokens attend to audio tokens
        return s.head(s.an(hv + att * m[:, None, None]).mean(1))  # no audio -> attention output is switched off

def make_data(mask):
    idx = np.where(mask)[0]; sp = split[idx]
    tr = idx[sp == "train"]
    mu_v, sd_v = V[tr].reshape(-1, 768).mean(0), V[tr].reshape(-1, 768).std(0) + 1e-6
    ta = tr[aud_ok[tr] == 1]
    mu_a, sd_a = A[ta].reshape(-1, 768).mean(0), A[ta].reshape(-1, 768).std(0) + 1e-6
    T = lambda x: torch.tensor(x, dtype=torch.float32, device=dev)
    D = dict(V=T((V - mu_v) / sd_v), A=T((A - mu_a) / sd_a), M=T(aud_ok), G=T(G),
             y=torch.tensor(y, device=dev), y_np=y)
    for s in ("train", "val", "test"):
        D[s] = torch.tensor(idx[sp == s], device=dev); D[s + "_np"] = idx[sp == s]
    # "wrong audio" control: for each test clip, audio from a random clip of a different class
    rng = np.random.default_rng(123); te = D["test_np"]; yt = y[te]
    D["wrong"] = torch.tensor([rng.choice(np.where(yt != yt[i])[0]) for i in range(len(te))], device=dev)
    return D

def train_one(mode, D, seed, mod_drop):
    torch.manual_seed(seed); np.random.seed(seed)
    net = Net(mode, CFG["d"], CFG["drop"]).to(dev)
    tr = D["train"]; ytr = D["y"][tr]
    cnt = torch.bincount(ytr, minlength=4).float(); w = (len(ytr) / (4 * cnt.clamp(min=1))).to(dev)
    opt = torch.optim.AdamW(net.parameters(), lr=CFG["lr"], weight_decay=CFG["wd"])
    steps = CFG["epochs"] * int(np.ceil(len(tr) / CFG["bs"]))
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps)
    lossf = nn.CrossEntropyLoss(weight=w, label_smoothing=CFG["smooth"])
    for ep in range(CFG["epochs"]):
        net.train(); perm = torch.randperm(len(tr), device=dev)
        for i in range(0, len(tr), CFG["bs"]):
            ix = tr[perm[i:i + CFG["bs"]]]
            m = D["M"][ix]
            if mod_drop > 0 and mode in ("early", "gate", "xattn"):
                m = m * (torch.rand(len(ix), device=dev) >= mod_drop).float()   # randomly hide the audio
            loss = lossf(net(D["V"][ix], D["A"][ix], m, D["G"][ix]), D["y"][ix])
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    return net.eval()

@torch.no_grad()
def predict(net, D, part, cond="real"):
    ix = D[part]; v, a, m, g = D["V"][ix], D["A"][ix], D["M"][ix], D["G"][ix]
    if cond == "none":
        m = torch.zeros_like(m)
    elif cond == "shuffle":                                    # audio (and gate inputs) of a different-class clip
        j = D["wrong"]; a, m, g = a[j], m[j], g[j]
    return torch.softmax(net(v, a, m, g), 1).cpu().numpy()

@torch.no_grad()
def gate_values(net, D):
    ix = D["test"]; return net.gate_value(D["M"][ix], D["G"][ix]).cpu().numpy()

In [4]:
# Cell 4: smoke test (fails fast if something is wrong)
for mode in ("video", "audio", "early", "gate", "xattn"):
    net = Net(mode).to(dev)
    v, a, g = torch.randn(6, 8, 768, device=dev), torch.randn(6, 5, 768, device=dev), torch.rand(6, 3, device=dev)
    for m in (torch.ones(6, device=dev), torch.zeros(6, device=dev)):          # audio present, audio absent
        out = net(v, a, m, g); assert out.shape == (6, 4) and torch.isfinite(out).all(), mode
        out.sum().backward()
print("all five model modes run forward and backward, with and without audio")

all five model modes run forward and backward, with and without audio


In [5]:
# Cell 5: run all methods on one set
SPECS = [("video only", "video", 0), ("audio only", "audio", 0), ("early fusion", "early", CFG["mod_drop"]),
         ("gated fusion (router)", "gate", CFG["mod_drop"]), ("cross-attention", "xattn", CFG["mod_drop"]),
         ("gated, no mod-drop", "gate", 0), ("cross-attention, no mod-drop", "xattn", 0)]

def run_set(name, mask):
    D = make_data(mask); te = D["test_np"]; yt = y[te]
    print(f"\n##### {name}: train={len(D['train'])} val={len(D['val'])} test={len(te)} #####")
    P, gv = {}, {}
    t0 = time.time()
    for label, mode, md in SPECS:
        conds = ["real"] if mode == "video" else (["real", "shuffle"] if mode == "audio" else ["real", "none", "shuffle"])
        acc = {("real", "val"): []}; acc.update({(c, "test"): [] for c in conds}); gl = []
        for s_ in CFG["seeds"]:
            net = train_one(mode, D, s_, md)
            acc[("real", "val")].append(predict(net, D, "val", "real"))
            for c in conds: acc[(c, "test")].append(predict(net, D, "test", c))
            if mode == "gate": gl.append(gate_values(net, D))
        for k, v in acc.items(): P[(label, *k)] = np.mean(v, 0)
        if gl: gv[label] = np.mean(gl, 0)
        print(f"  {label:30s} done ({time.time() - t0:.0f}s)")
    # late fusion: combine the video-only and audio-only probabilities, weight chosen on val
    yv = y[D["val_np"]]; best = (-1, 0)
    for w in np.arange(0, 1.55, 0.1):
        lp = np.log(P[("video only", "real", "val")] + 1e-8) + w * np.log(P[("audio only", "real", "val")] + 1e-8)
        f = f1_score(yv, lp.argmax(1), average="macro")
        if f > best[0] + 1e-9: best = (f, w)
    w = best[1]; lab = f"late fusion (w={w:.1f})"
    for c in ("real", "none", "shuffle"):
        lp = np.log(P[("video only", "real", "test")] + 1e-8)
        if c != "none": lp = lp + w * np.log(P[("audio only", c, "test")] + 1e-8)
        e = np.exp(lp - lp.max(1, keepdims=True)); P[(lab, c, "test")] = e / e.sum(1, keepdims=True)
    return D, P, gv, yt, [s[0] for s in SPECS] + [lab]

In [6]:
# Cell 6: results tables
def report(name, D, P, gv, yt, methods):
    te = D["test_np"]; base = P[("video only", "real", "test")]; rows = []
    for m in methods:
        pt, ci = with_ci(yt, P[(m, "real", "test")])
        dm = "" if m == "video only" else "{:+.3f} [{:+.3f}, {:+.3f}]".format(*paired(yt, base, P[(m, "real", "test")]))
        rows.append(dict(method=m, macro_F1=fmt(pt, ci, "macro_F1"), macro_AUC=fmt(pt, ci, "macro_AUC"),
                         top2=f"{pt['top2_acc']:.3f}", diff_vs_video=dm, **{c: round(pt[f"rec_{c}"], 2) for c in CLASS_NAMES}))
    main = pd.DataFrame(rows); pd.set_option("display.width", 250, "display.max_colwidth", 50)
    print(f"\n===== {name}: main results (mean of {len(CFG['seeds'])} seeds; diff = macro-F1 minus video only) =====")
    print(main.to_string(index=False)); main.to_csv(f"{RES}/fusion_main_{name}.csv", index=False)

    # controls: is the audio actually used?
    rows = []
    for m in methods:
        if m in ("video only",): continue
        r = dict(method=m)
        for c in ("real", "none", "shuffle"):
            if (m, c, "test") in P: r[f"audio: {c}"] = round(f1_score(yt, P[(m, c, "test")].argmax(1), average="macro"), 3)
        rows.append(r)
    ctl = pd.DataFrame(rows); print(f"\n{name}: controls (macro-F1). real = normal, none = audio removed, shuffle = audio from a different-class clip")
    print(ctl.to_string(index=False)); ctl.to_csv(f"{RES}/fusion_controls_{name}.csv", index=False)

    # by audio type
    g = grp[te]; rows = []
    for gname in ("music-dominant", "speech present", "other audio"):
        sel = g == gname
        if sel.sum() < 20: continue
        r = dict(audio_type=gname, n=int(sel.sum()))
        for m in ["video only", "audio only", "early fusion", "gated fusion (router)", "cross-attention", methods[-1]]:
            r[m] = round(f1_score(yt[sel], P[(m, "real", "test")][sel].argmax(1), average="macro"), 3)
        rows.append(r)
    byg = pd.DataFrame(rows); print(f"\n{name}: macro-F1 by audio type (small groups: treat differences of a few points as noise)")
    print(byg.to_string(index=False)); byg.to_csv(f"{RES}/fusion_by_audio_type_{name}.csv", index=False)

    # what did the router learn?
    if "gated fusion (router)" in gv:
        gvals = gv["gated fusion (router)"]
        print(f"\n{name}: mean gate value (audio weight) on test clips, by audio type")
        print(pd.Series({k: round(float(gvals[g == k].mean()), 3) for k in ("music-dominant", "speech present", "other audio") if (g == k).any()}).to_string())

D_m, P_m, gv_m, yt_m, meth_m = run_set("matched", in_matched)
report("matched", D_m, P_m, gv_m, yt_m, meth_m)


##### matched: train=1004 val=104 test=395 #####
  video only                     done (11s)
  audio only                     done (18s)
  early fusion                   done (27s)
  gated fusion (router)          done (38s)
  cross-attention                done (53s)
  gated, no mod-drop             done (64s)
  cross-attention, no mod-drop   done (79s)

===== matched: main results (mean of 5 seeds; diff = macro-F1 minus video only) =====
                      method             macro_F1            macro_AUC  top2           diff_vs_video  Safe  Harmful  Adult  Suicide
                  video only 0.841 [0.805, 0.875] 0.947 [0.928, 0.965] 0.922                          0.93     0.72   0.82     0.91
                  audio only 0.563 [0.510, 0.609] 0.791 [0.760, 0.820] 0.782 -0.279 [-0.332, -0.222]  0.48     0.47   0.57     0.74
                early fusion 0.847 [0.810, 0.880] 0.951 [0.933, 0.968] 0.924 +0.006 [-0.011, +0.022]  0.92     0.75   0.81     0.92
       gated fusion (router

In [7]:
# Cell 7: secondary set (full clean set)
D_f, P_f, gv_f, yt_f, meth_f = run_set("full", np.ones(N, bool))
report("full", D_f, P_f, gv_f, yt_f, meth_f)
print(f"\nSaved tables to {RES}")


##### full: train=2684 val=336 test=870 #####
  video only                     done (18s)
  audio only                     done (37s)
  early fusion                   done (62s)
  gated fusion (router)          done (91s)
  cross-attention                done (129s)
  gated, no mod-drop             done (156s)
  cross-attention, no mod-drop   done (194s)

===== full: main results (mean of 5 seeds; diff = macro-F1 minus video only) =====
                      method             macro_F1            macro_AUC  top2           diff_vs_video  Safe  Harmful  Adult  Suicide
                  video only 0.854 [0.829, 0.877] 0.965 [0.955, 0.974] 0.940                          0.92     0.74   0.82     0.92
                  audio only 0.664 [0.633, 0.695] 0.869 [0.852, 0.889] 0.837 -0.190 [-0.225, -0.157]  0.72     0.56   0.59     0.79
                early fusion 0.864 [0.841, 0.885] 0.971 [0.962, 0.979] 0.944 +0.010 [-0.002, +0.022]  0.92     0.74   0.83     0.95
       gated fusion (router) 0

In [8]:
!zip -r results_new.zip /kaggle/working/results

  adding: kaggle/working/results/ (stored 0%)
  adding: kaggle/working/results/fusion_main_matched.csv (deflated 58%)
  adding: kaggle/working/results/fusion_controls_full.csv (deflated 43%)
  adding: kaggle/working/results/fusion_by_audio_type_full.csv (deflated 36%)
  adding: kaggle/working/results/fusion_controls_matched.csv (deflated 44%)
  adding: kaggle/working/results/fusion_by_audio_type_matched.csv (deflated 36%)
  adding: kaggle/working/results/fusion_main_full.csv (deflated 57%)
